# Dataset Consolidation — antigen_predictor

Construye **dos versiones** del dataset para entrenamiento:

| Archivo | Descripción |
|---|---|
| `dataset_human.csv` | Solo huésped humano (referencia clínica) |
| `dataset_extended.csv` | Humano + ratón + primates (mayor volumen) |

## Fixes aplicados en esta versión
- **`get_amino_acids_percent()` → `pa.amino_acids_percent`**: en Biopython ≥ 1.80 es una
  propiedad (dict), no un método. Usar `pa.amino_acids_percent` sin paréntesis.
- **Negativos reales añadidos**: proteínas del proteoma de los patógenos sin evidencia
  de antigenicidad en IEDB, recuperadas directamente de UniProt.
- **Dataset ampliado**: el catálogo IEDB contiene solo proteínas ya estudiadas
  (todas positivas), por lo que ampliar el host no genera nuevos negativos.
  Los negativos se obtienen de proteínas fuera del catálogo IEDB.

## 0. Imports y configuración

In [1]:
import pandas as pd
import numpy as np
import re
import time
import requests
from pathlib import Path
from Bio.SeqUtils.ProtParam import ProteinAnalysis

RAW_PATH       = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

TARGET_TAXON_PATTERNS = [
    "NCBITaxon_2697049",   # SARS-CoV-2
    "10002316",            # SARS-CoV-1 (ID interno IEDB)
    "NCBITaxon_11320",     # Influenza A
]
TAXON_LABEL_MAP = [
    ("NCBITaxon_2697049", "SARS-CoV-2"),
    ("10002316",          "SARS-CoV-1"),
    ("NCBITaxon_11320",   "Influenza_A"),
]

# Hosts para dataset estricto
HOSTS_STRICT = ["NCBITaxon_9606"]

# Hosts para dataset ampliado
# Humano + roedores (modelo animal principal) + primates no humanos
# ONTIE_ = cepas inbred de ratón en IEDB (BALB/c, C57BL/6, etc.)
HOSTS_EXTENDED = [
    "NCBITaxon_9606",   # Homo sapiens
    "NCBITaxon_10090",  # Mus musculus
    "NCBITaxon_10116",  # Rattus norvegicus
    "NCBITaxon_9544",   # Macaca mulatta
    "NCBITaxon_9483",   # Callithrix jacchus
    "NCBITaxon_9986",   # Oryctolagus cuniculus (conejo)
    "ONTIE_",           # Cepas inbred de raton (BALB/c, C57BL/6...)
]

STANDARD_AA = list("ACDEFGHIKLMNPQRSTVWY")

# Negativos reales: proteinas de los patogenos objetivo sin evidencia en IEDB
# Fuente: proteomas de referencia UniProt
# Justificacion: estas proteinas existen en el virus pero no tienen epitopos
# documentados -> candidatos validos como label=0
EXTRA_NEGATIVES = [
    # SARS-CoV-2: NSPs con muy pocos epitopos documentados
    {"uniprot_id": "P0DTD7",  "antigen_name": "Non-structural protein nsp1",       "pathogen": "SARS-CoV-2"},
    {"uniprot_id": "P0DTD5",  "antigen_name": "Non-structural protein nsp2",       "pathogen": "SARS-CoV-2"},
    # SARS-CoV-1: proteina accesoria con poca evidencia experimental
    {"uniprot_id": "P59638",  "antigen_name": "ORF9b-like protein",                "pathogen": "SARS-CoV-1"},
    # Influenza A: NS2/NEP (nuclear export protein 2) - muy pocos datos humanos
    {"uniprot_id": "P03498",  "antigen_name": "Non-structural protein 2 NS2/NEP",  "pathogen": "Influenza_A"},
]

print("OK")
print(f"Raw:       {RAW_PATH.resolve()}")
print(f"Processed: {PROCESSED_PATH.resolve()}")

OK
Raw:       C:\Saturdays\aigenix\data\raw
Processed: C:\Saturdays\aigenix\data\processed


## Funciones auxiliares

In [2]:
def extract_uniprot_id(iri_series):
    def _e(iri):
        if pd.isna(iri): return np.nan
        s = str(iri)
        if "uniprot.org/uniprot/" not in s: return np.nan
        code = s.rstrip("/").split("/")[-1]
        return re.sub(r"\.\d+$", "", code) or np.nan
    return iri_series.apply(_e)

def taxon_match(s):
    return s.astype(str).str.contains("|".join(TARGET_TAXON_PATTERNS), na=False)

def host_match(s, patterns):
    return s.astype(str).str.contains("|".join(patterns), na=False)

def get_taxon_label(iri):
    s = str(iri)
    for pat, label in TAXON_LABEL_MAP:
        if pat in s: return label
    return "Unknown"

def load_iedb_csv(filepath):
    row0 = pd.read_csv(filepath, nrows=1, header=None).iloc[0].tolist()
    ffill = []
    last = ""
    for v in row0:
        if pd.notna(v) and str(v).strip(): last = str(v).strip()
        ffill.append(last)
    df = pd.read_csv(filepath, header=1, low_memory=False)
    return df, ffill

def find_col(ffill, cols, group, sub):
    for g, c in zip(ffill, cols):
        if group in str(g) and sub in str(c): return c
    return None

def find_col_fallback(cols, sub):
    return next((c for c in cols if sub in str(c)), None)

def find_host_iri_col(df, cols):
    for c in cols:
        if re.sub(r"\.\d+$", "", str(c)) == "IRI":
            samp = df[c].dropna().astype(str).head(500)
            if (samp.str.contains(r"NCBITaxon_\d+", na=False).any() and
                samp.str.contains("NCBITaxon_9606", na=False).any()):
                return c
    return None


def compute_features(sequence):
    """
    Calcula 26 features de secuencia usando Biopython.

    NOTA API Biopython >= 1.80:
      - pa.amino_acids_percent  -> propiedad dict (SIN parentesis)
      - pa.get_amino_acids_percent() fue eliminado en Biopython 1.80+
    """
    if sequence is None or not isinstance(sequence, str) or not sequence.strip():
        return None
    try:
        clean = "".join(aa for aa in sequence.upper() if aa in STANDARD_AA)
        if len(clean) < 10:
            return None
        pa = ProteinAnalysis(clean)
        feats = {
            "length":            len(clean),
            "molecular_weight":  pa.molecular_weight(),
            "isoelectric_point": pa.isoelectric_point(),
            "gravy":             pa.gravy(),
            "instability_index": pa.instability_index(),
            "aromaticity":       pa.aromaticity(),
        }
        # CORRECTO para Biopython >= 1.80: propiedad, no metodo
        aa_pct = pa.amino_acids_percent
        for aa in STANDARD_AA:
            feats[f"aa_{aa}"] = aa_pct.get(aa, 0.0)
        return feats
    except Exception as e:
        print(f"  Error compute_features({sequence[:20]}...): {e}")
        return None


def fetch_sequence(uniprot_id, retries=3):
    if not uniprot_id or pd.isna(uniprot_id): return None
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                lines = r.text.strip().split("\n")
                seq = "".join(l for l in lines if not l.startswith(">"))
                return seq.strip() or None
            elif r.status_code == 404:
                return None
            time.sleep(1)
        except Exception:
            time.sleep(2 * (attempt + 1))
    return None


def get_positives(assay_df, species_col, host_col, qual_col, molpar_col, host_patterns, tag):
    """Filtra un dataframe de ensayos y devuelve set de UniProt IDs positivos."""
    f = assay_df[taxon_match(assay_df[species_col])]
    f = f[host_match(f[host_col], host_patterns)]
    f = f[f[qual_col].astype(str).str.contains("Positive", na=False)]
    f = f[f[molpar_col].astype(str).str.contains("uniprot.org/uniprot/", na=False)]
    f = f.copy()
    f["_uid"] = extract_uniprot_id(f[molpar_col])
    pos = set(f["_uid"].dropna().unique())
    print(f"  [{tag}] {len(assay_df):,} -> {len(pos)} proteinas positivas unicas")
    return pos


print("OK — funciones definidas")
# Test rapido compute_features
_t = compute_features("MKTIIALSYIFCLVFAQK")
print(f"Test compute_features: {'OK (' + str(len(_t)) + ' features)' if _t else 'FALLO'}")

OK — funciones definidas
Test compute_features: OK (26 features)


## FASE 1 — antigen_full_v3

In [3]:
print("Cargando antigen_full_v3.csv...")
antigen_raw = pd.read_csv(RAW_PATH / "antigen_full_v3.csv", header=1, low_memory=False)
antigen_raw.columns = ["antigen_name","antigen_iri","organism_name","organism_iri",
                       "n_epitopes","n_assays","n_references"]
print(f"Filas: {len(antigen_raw):,}")

af = antigen_raw[taxon_match(antigen_raw["organism_iri"])].copy()
af = af[af["antigen_iri"].astype(str).str.contains("uniprot.org/uniprot/", na=False)]
af = af[~af["antigen_name"].astype(str).str.startswith("Two components")]
af["uniprot_id"] = extract_uniprot_id(af["antigen_iri"])
af["pathogen"]   = af["organism_iri"].apply(get_taxon_label)
af = af.drop_duplicates(subset="uniprot_id", keep="first")

proteins_catalog = af[["uniprot_id","antigen_name","pathogen","n_epitopes","n_assays"]].reset_index(drop=True)

print(f"Proteinas del catalogo IEDB: {len(proteins_catalog)}")
print(proteins_catalog["pathogen"].value_counts().to_string())

Cargando antigen_full_v3.csv...
Filas: 86,292
Proteinas del catalogo IEDB: 42
pathogen
SARS-CoV-2     16
SARS-CoV-1     15
Influenza_A    11


## FASE 2 — Cargar tcell y bcell

In [4]:
print("Cargando tcell_full_v3.csv...")
tcell_raw, tc_row0 = load_iedb_csv(RAW_PATH / "tcell_full_v3.csv")
print(f"Filas: {len(tcell_raw):,}")

tc_s = find_col(tc_row0, tcell_raw.columns, "Epitope", "Species IRI")
tc_h = find_col(tc_row0, tcell_raw.columns, "Host", "IRI") or find_host_iri_col(tcell_raw, tcell_raw.columns)
tc_q = find_col(tc_row0, tcell_raw.columns, "Assay", "Qualitative") or find_col_fallback(tcell_raw.columns, "Qualitative")
tc_m = find_col(tc_row0, tcell_raw.columns, "Epitope", "Molecule Parent IRI") or find_col_fallback(tcell_raw.columns, "Molecule Parent IRI")

if tc_s is None: tc_s = find_col_fallback(tcell_raw.columns, "Species IRI")

assert all(c is not None for c in [tc_s,tc_h,tc_q,tc_m]), f"Columnas tcell no encontradas: s={tc_s} h={tc_h} q={tc_q} m={tc_m}"
print(f"  Species IRI: '{tc_s}' | Host IRI: '{tc_h}' | Qual: '{tc_q}' | MolPar: '{tc_m}'")

Cargando tcell_full_v3.csv...
Filas: 569,577
  Species IRI: 'Species IRI' | Host IRI: 'IRI.2' | Qual: 'Qualitative Measurement' | MolPar: 'Molecule Parent IRI'


In [5]:
print("Cargando bcell_full_v3.csv...")
bcell_raw, bc_row0 = load_iedb_csv(RAW_PATH / "bcell_full_v3.csv")
print(f"Filas: {len(bcell_raw):,}")

bc_s = find_col(bc_row0, bcell_raw.columns, "Epitope", "Species IRI")
bc_h = find_col(bc_row0, bcell_raw.columns, "Host", "IRI") or find_host_iri_col(bcell_raw, bcell_raw.columns)
bc_q = find_col(bc_row0, bcell_raw.columns, "Assay", "Qualitative") or find_col_fallback(bcell_raw.columns, "Qualitative")
bc_m = find_col(bc_row0, bcell_raw.columns, "Epitope", "Molecule Parent IRI") or find_col_fallback(bcell_raw.columns, "Molecule Parent IRI")

if bc_s is None: bc_s = find_col_fallback(bcell_raw.columns, "Species IRI")

assert all(c is not None for c in [bc_s,bc_h,bc_q,bc_m]), f"Columnas bcell no encontradas: s={bc_s} h={bc_h} q={bc_q} m={bc_m}"
print(f"  Species IRI: '{bc_s}' | Host IRI: '{bc_h}' | Qual: '{bc_q}' | MolPar: '{bc_m}'")

print("\nHosts mas frecuentes en bcell (top 8):")
print(bcell_raw[bc_h].value_counts().head(8).to_string())

Cargando bcell_full_v3.csv...
Filas: 1,436,588
  Species IRI: 'Species IRI' | Host IRI: 'IRI.2' | Qual: 'Qualitative Measure' | MolPar: 'Molecule Parent IRI'

Hosts mas frecuentes en bcell (top 8):
IRI.2
http://purl.obolibrary.org/obo/NCBITaxon_9606       1229349
https://ontology.iedb.org/ontology/ONTIE_0000001      53945
http://purl.obolibrary.org/obo/NCBITaxon_9986         20235
http://purl.obolibrary.org/obo/NCBITaxon_10090        19983
https://ontology.iedb.org/ontology/ONTIE_0000126      17171
https://ontology.iedb.org/ontology/ONTIE_0000062      16873
http://purl.obolibrary.org/obo/NCBITaxon_9544         15742
http://purl.obolibrary.org/obo/NCBITaxon_9823          3065


## FASE 3 — Extraer positivos (strict y extended)

In [6]:
print("=== Positivos ESTRICTO (solo humano) ===")
tc_strict = get_positives(tcell_raw, tc_s, tc_h, tc_q, tc_m, HOSTS_STRICT, "tcell-strict")
bc_strict = get_positives(bcell_raw, bc_s, bc_h, bc_q, bc_m, HOSTS_STRICT, "bcell-strict")
all_strict = tc_strict | bc_strict
print(f"Union: {len(all_strict)} proteinas positivas")

print("\n=== Positivos AMPLIADO (humano + raton + primates) ===")
tc_ext = get_positives(tcell_raw, tc_s, tc_h, tc_q, tc_m, HOSTS_EXTENDED, "tcell-ext")
bc_ext = get_positives(bcell_raw, bc_s, bc_h, bc_q, bc_m, HOSTS_EXTENDED, "bcell-ext")
all_ext = tc_ext | bc_ext
print(f"Union: {len(all_ext)} proteinas positivas")

print(f"\nGanados al ampliar hosts: {len(all_ext - all_strict)}")
if all_ext - all_strict:
    print(sorted(all_ext - all_strict))
else:
    print("  -> El catalogo IEDB contiene solo proteinas con epitopos estudiados.")
    print("     Los negativos deben venir del proteoma completo (fuera del catalogo).")

=== Positivos ESTRICTO (solo humano) ===
  [tcell-strict] 569,577 -> 31 proteinas positivas unicas
  [bcell-strict] 1,436,588 -> 39 proteinas positivas unicas
Union: 41 proteinas positivas

=== Positivos AMPLIADO (humano + raton + primates) ===
  [tcell-ext] 569,577 -> 32 proteinas positivas unicas
  [bcell-ext] 1,436,588 -> 39 proteinas positivas unicas
Union: 41 proteinas positivas

Ganados al ampliar hosts: 0
  -> El catalogo IEDB contiene solo proteinas con epitopos estudiados.
     Los negativos deben venir del proteoma completo (fuera del catalogo).


## FASE 4 — Recuperar secuencias

Recuperamos secuencias para el catálogo IEDB + negativos adicionales en una sola pasada.

In [7]:
# Combinar catálogo IEDB + negativos adicionales en una sola tabla
extra_df = pd.DataFrame(EXTRA_NEGATIVES)
extra_df["n_epitopes"] = 0
extra_df["n_assays"]   = 0

all_proteins = pd.concat(
    [proteins_catalog[["uniprot_id","antigen_name","pathogen","n_epitopes","n_assays"]],
     extra_df[["uniprot_id","antigen_name","pathogen","n_epitopes","n_assays"]]],
    ignore_index=True
)
# Eliminar duplicados por si algún negativo ya está en catálogo
all_proteins = all_proteins.drop_duplicates(subset="uniprot_id", keep="first")

print(f"Total proteinas (catalogo + negativos): {len(all_proteins)}")
print(f"  Del catalogo IEDB:         {len(proteins_catalog)}")
print(f"  Negativos adicionales:     {len(extra_df)}")
print(f"\nNEGATIVOS que vamos a añadir:")
print(extra_df[["uniprot_id","antigen_name","pathogen"]].to_string())

Total proteinas (catalogo + negativos): 46
  Del catalogo IEDB:         42
  Negativos adicionales:     4

NEGATIVOS que vamos a añadir:
  uniprot_id                      antigen_name     pathogen
0     P0DTD7       Non-structural protein nsp1   SARS-CoV-2
1     P0DTD5       Non-structural protein nsp2   SARS-CoV-2
2     P59638                ORF9b-like protein   SARS-CoV-1
3     P03498  Non-structural protein 2 NS2/NEP  Influenza_A


In [8]:
total = len(all_proteins)
print(f"Recuperando {total} secuencias desde UniProt (~{total*0.5:.0f}s)...\n")

sequences = {}
for i, row in all_proteins.iterrows():
    uid = row["uniprot_id"]
    seq = fetch_sequence(uid)
    sequences[uid] = seq
    status = f"{len(seq)} aa" if seq else "NOT FOUND"
    print(f"  [{i+1:3d}/{total}] {uid:15s} | {row['pathogen']:12s} | {status}")
    time.sleep(0.35)

all_proteins["sequence"] = all_proteins["uniprot_id"].map(sequences)

n_miss = all_proteins["sequence"].isna().sum()
print(f"\nCon secuencia:   {len(all_proteins) - n_miss}")
print(f"Sin secuencia:   {n_miss}")
if n_miss:
    print("Sin secuencia:")
    print(all_proteins[all_proteins["sequence"].isna()][["uniprot_id","antigen_name"]].to_string())

all_proteins = all_proteins[all_proteins["sequence"].notna()].copy()
all_proteins["seq_length"] = all_proteins["sequence"].str.len()
all_proteins = all_proteins[all_proteins["seq_length"] >= 10]
print(f"\nProteinas con secuencia valida (>= 10 aa): {len(all_proteins)}")

Recuperando 46 secuencias desde UniProt (~23s)...

  [  1/46] P59632          | SARS-CoV-1   | 274 aa
  [  2/46] P59633          | SARS-CoV-1   | 154 aa
  [  3/46] P59634          | SARS-CoV-1   | 63 aa
  [  4/46] P59635          | SARS-CoV-1   | 122 aa
  [  5/46] P59636          | SARS-CoV-1   | 98 aa
  [  6/46] P59637          | SARS-CoV-1   | 76 aa
  [  7/46] P59594          | SARS-CoV-1   | 1255 aa
  [  8/46] P59595          | SARS-CoV-1   | 422 aa
  [  9/46] P59596          | SARS-CoV-1   | 221 aa
  [ 10/46] P06821          | Influenza_A  | 97 aa
  [ 11/46] Q7TFA0          | SARS-CoV-1   | 39 aa
  [ 12/46] Q7TFA1          | SARS-CoV-1   | 44 aa
  [ 13/46] Q7TLC7          | SARS-CoV-1   | 70 aa
  [ 14/46] P0C0U1          | Influenza_A  | 87 aa
  [ 15/46] P03452          | Influenza_A  | 565 aa
  [ 16/46] P03468          | Influenza_A  | 454 aa
  [ 17/46] P03466          | Influenza_A  | 498 aa
  [ 18/46] P03485          | Influenza_A  | 252 aa
  [ 19/46] P03496          | Influenza

## FASE 5 — Feature Engineering

In [9]:
print(f"Calculando features para {len(all_proteins)} proteinas...")

feature_rows = []
failed = []

for _, row in all_proteins.iterrows():
    feats = compute_features(row["sequence"])
    if feats:
        feats["uniprot_id"]   = row["uniprot_id"]
        feats["antigen_name"] = row["antigen_name"]
        feats["pathogen"]     = row["pathogen"]
        feature_rows.append(feats)
    else:
        failed.append(row["uniprot_id"])
        print(f"  FALLO features: {row['uniprot_id']} (seq: {row['sequence'][:30]}...)")

print(f"\nFeatures OK: {len(feature_rows)} | Fallidas: {len(failed)}")
if failed:
    print(f"IDs fallidos: {failed}")

assert len(feature_rows) > 0, "ERROR: No se calcularon features. Revisar compute_features."

features_df = pd.DataFrame(feature_rows)
meta_cols   = ["uniprot_id", "antigen_name", "pathogen"]
feat_cols   = [c for c in features_df.columns if c not in meta_cols]

print(f"Features calculadas por proteina: {len(feat_cols)}")
print("Features:", feat_cols[:8], "...", feat_cols[-3:])

Calculando features para 46 proteinas...

Features OK: 46 | Fallidas: 0
Features calculadas por proteina: 26
Features: ['length', 'molecular_weight', 'isoelectric_point', 'gravy', 'instability_index', 'aromaticity', 'aa_A', 'aa_C'] ... ['aa_V', 'aa_W', 'aa_Y']


## FASE 6 — Asignar etiquetas y construir los dos datasets

In [10]:
neg_ids = {neg["uniprot_id"] for neg in EXTRA_NEGATIVES}

def make_dataset(positives_set, label_name):
    """Ensambla el dataset asignando labels y reordenando columnas."""
    df = features_df.copy()
    # label=1 si esta en positivos Y no es un negativo explicito
    # label=0 si es negativo explicito O no esta en positivos
    df["label"] = df["uniprot_id"].apply(
        lambda uid: 0 if uid in neg_ids else (1 if uid in positives_set else 0)
    )
    df = df[meta_cols + feat_cols + ["label"]]
    vc = df["label"].value_counts()
    print(f"\n[{label_name}] Shape: {df.shape}")
    print(f"  label=1: {vc.get(1,0)} | label=0: {vc.get(0,0)} | "
          f"balance: {vc.get(1,0)/len(df):.1%} positivos")
    print(df.groupby(["pathogen","label"]).size().unstack(fill_value=0).to_string())
    return df


print("=== DATASET ESTRICTO (solo humano) ===")
dataset_human = make_dataset(all_strict, "human")

print("\n=== DATASET AMPLIADO (humano + raton + primates) ===")
dataset_ext = make_dataset(all_ext, "extended")

=== DATASET ESTRICTO (solo humano) ===

[human] Shape: (46, 30)
  label=1: 41 | label=0: 5 | balance: 89.1% positivos
label        0   1
pathogen          
Influenza_A  1  11
SARS-CoV-1   1  15
SARS-CoV-2   3  15

=== DATASET AMPLIADO (humano + raton + primates) ===

[extended] Shape: (46, 30)
  label=1: 41 | label=0: 5 | balance: 89.1% positivos
label        0   1
pathogen          
Influenza_A  1  11
SARS-CoV-1   1  15
SARS-CoV-2   3  15


## FASE 7 — Verificación y guardado

In [11]:
def verify_and_save(df, filename, label):
    print(f"\n--- {label} ---")
    print(f"  Shape:           {df.shape}")
    print(f"  NaN totales:     {df.isnull().sum().sum()}")
    print(f"  Duplicados uid:  {df['uniprot_id'].duplicated().sum()}")
    vc = df["label"].value_counts()
    print(f"  label=1:         {vc.get(1,0)}")
    print(f"  label=0:         {vc.get(0,0)}")
    print(f"  Balance:         {vc.get(1,0)/len(df):.1%} positivos")
    path = PROCESSED_PATH / filename
    df.to_csv(path, index=False)
    print(f"  Guardado:        {path.resolve()}")
    print(f"  Tamano:          {path.stat().st_size/1024:.1f} KB")


print("=== VERIFICACION Y GUARDADO ===")
verify_and_save(dataset_human, "dataset_human.csv",    "Dataset estricto (humano)")
verify_and_save(dataset_ext,   "dataset_extended.csv", "Dataset ampliado (humano + raton + primates)")

# Auditorias
for df, fname in [(dataset_human, "proteins_labeled_human.csv"),
                  (dataset_ext,   "proteins_labeled_extended.csv")]:
    df[["uniprot_id","antigen_name","pathogen","label"]].to_csv(
        PROCESSED_PATH / fname, index=False)

print("\n=== ARCHIVOS GENERADOS ===")
for f in sorted(PROCESSED_PATH.glob("*.csv")):
    print(f"  {f.name}: {f.stat().st_size/1024:.1f} KB")

print("\n=== RESUMEN METODOLOGICO ===")
print("  dataset_human.csv    -> evaluacion clinica final (solo humano)")
print("  dataset_extended.csv -> entrenamiento (mayor volumen)")
print("  Negativos: proteinas del proteoma de los patogenos sin")
print("  epitopos documentados en IEDB (label=0 es ausencia de evidencia)")
print("\nPipeline completado correctamente.")

=== VERIFICACION Y GUARDADO ===

--- Dataset estricto (humano) ---
  Shape:           (46, 30)
  NaN totales:     0
  Duplicados uid:  0
  label=1:         41
  label=0:         5
  Balance:         89.1% positivos
  Guardado:        C:\Saturdays\aigenix\data\processed\dataset_human.csv
  Tamano:          21.9 KB

--- Dataset ampliado (humano + raton + primates) ---
  Shape:           (46, 30)
  NaN totales:     0
  Duplicados uid:  0
  label=1:         41
  label=0:         5
  Balance:         89.1% positivos
  Guardado:        C:\Saturdays\aigenix\data\processed\dataset_extended.csv
  Tamano:          21.9 KB

=== ARCHIVOS GENERADOS ===
  dataset_extended.csv: 21.9 KB
  dataset_human.csv: 21.9 KB
  proteins_labeled.csv: 2.8 KB
  proteins_labeled_extended.csv: 2.6 KB
  proteins_labeled_human.csv: 2.6 KB

=== RESUMEN METODOLOGICO ===
  dataset_human.csv    -> evaluacion clinica final (solo humano)
  dataset_extended.csv -> entrenamiento (mayor volumen)
  Negativos: proteinas del prote